# ==============================================================================
# 🏠 URBAN PROPERTY VALUATION: REAL ESTATE VIA SUPPORT VECTOR REGRESSOR
# ==============================================================================
### Core Principles:
Forecasting house unit price based on geographical coordinates (`latitude`, `longitude`), proximity to public transit (`distance_to_mrt`), age, and local amenities.

SVR's $\varepsilon$-tube naturally accommodates idiosyncratic real estate negotiations while learning smooth geographic spatial gradients.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Real Estate Valuation Environment ready.")


✅ STEP 1: Real Estate Valuation Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Real Estate dataset (414 property sales).


In [2]:
# STEP 2 & 3: LOAD & HARMONIZE
data_path = 'data/real_estate/Real estate.csv'
df = pd.read_csv(data_path)

if 'No' in df.columns:
    df = df.drop(columns=['No'])

# Clean column headers
df.columns = [
    'transaction_date', 'house_age', 'distance_to_mrt', 
    'convenience_stores', 'latitude', 'longitude', 'unit_price'
]

print(f"📊 Dataset Shape: {df.shape[0]} property transactions, {df.shape[1]} attributes")
print(f"Unit Price Summary:\n{df['unit_price'].describe().round(2)}")
df.head(3)


📊 Dataset Shape: 414 property transactions, 7 attributes
Unit Price Summary:
count    414.00
mean      37.98
std       13.61
min        7.60
25%       27.70
50%       38.45
75%       46.60
max      117.50
Name: unit_price, dtype: float64


,transaction_date,house_age,distance_to_mrt,convenience_stores,latitude,longitude,unit_price
0,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
1,2012.917,19.5,306.59470,9,24.98034,121.53951,42.2
2,2013.583,13.3,561.98450,5,24.98746,121.54391,47.3


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `unit_price`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['unit_price'])
y = df['unit_price']

print(f"Missing attributes: {X.isnull().sum().sum()}")


Missing attributes: 0


## 🔒 STEP 6: TRAIN-TEST SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Properties: {X_train.shape[0]}")
print(f"Test Properties : {X_test.shape[0]}")


Train Properties: 331
Test Properties : 83


## ⚙️ STEP 7: DIRECT SCALE-STANDARDIZED PIPELINE
Standardizing spatial distances and coordinates.


In [5]:
# STEP 7: PIPELINE
pipe_svr = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=10, epsilon=2.0))
])
pipe_svr.fit(X_train, y_train)

r2_base = r2_score(y_test, pipe_svr.predict(X_test))
print(f"Baseline SVR R²  : {r2_base*100:.2f}%")
print(f"Baseline MAE     : {mean_absolute_error(y_test, pipe_svr.predict(X_test)):.2f} unit price")


Baseline SVR R²  : 76.76%
Baseline MAE     : 4.25 unit price


## 🎯 STEP 8 & 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing $C$, $\varepsilon$, and $\gamma$.


In [6]:
# STEP 8 & 9: GRID SEARCH
param_grid = {
    'svr__C': [5, 10, 25, 50],
    'svr__epsilon': [1.0, 2.0, 3.0],
    'svr__gamma': ['scale', 0.05, 0.1, 0.2]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf'))
])

grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Real Estate Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²: {grid.best_score_*100:.2f}%")


🏆 Best Real Estate Hyperparameters: {'svr__C': 50, 'svr__epsilon': 3.0, 'svr__gamma': 0.05}
🎯 Best 5-Fold CV R²: 65.14%


## 📊 STEP 10 & 11: VALUATION REPORT & RESIDUALS
Diagnostics on spatial property errors.


In [7]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📋 REAL ESTATE VALUATION REPORT:")
print(f"   Mean Absolute Error (MAE) : {mae:.2f} unit price")
print(f"   Root Mean Squared Error   : {rmse:.2f} unit price")
print(f"   R² Score                  : {r2 * 100:.2f}%")


📋 REAL ESTATE VALUATION REPORT:
   Mean Absolute Error (MAE) : 4.19 unit price
   Root Mean Squared Error   : 6.02 unit price
   R² Score                  : 78.38%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & MORTGAGE SMOKE TEST
Deploying the pipeline and testing mortgage instant appraisal latency.


In [8]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/real_estate_svr_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_property = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_val = loaded_pipe.predict(sample_property)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏡 LIVE AUTOMATED VALUATION (AVM):")
print(f"   Appraised Unit Price : 🏷️ {pred_val:.2f}")
print(f"   Actual Listed Value  : {y_test.iloc[0]:.2f}")
print(f"   Appraisal Latency    : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/real_estate_svr_pipeline.joblib (7,878 bytes)

🏡 LIVE AUTOMATED VALUATION (AVM):
   Appraised Unit Price : 🏷️ 51.93
   Actual Listed Value  : 45.10
   Appraisal Latency    : 6.692 ms (SLA < 2.0ms: CHECK)
